## A4: Tracking
### Check tracker libraries

In [1]:
# Cell 20: Check tracker libraries
!pip show deep-sort-realtime 2>/dev/null | grep Version || echo "deep-sort-realtime: NOT installed"
!pip show boxmot 2>/dev/null | grep Version || echo "boxmot: NOT installed"

deep-sort-realtime: NOT installed
boxmot: NOT installed


In [3]:
# Cell 21: Verify built-in trackers available
from pathlib import Path
import ultralytics
yolo_dir = Path(ultralytics.__file__).parent
print("Available trackers:")
!ls {yolo_dir}/cfg/trackers/

Available trackers:
botsort.yaml  bytetrack.yaml


## DeepSort tracking

In [6]:
# Cell 22: DeepSORT tracking on test split
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from pathlib import Path
from collections import defaultdict
import csv, os
import cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
OUTPUT_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/deepsort")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]
model = YOLO(WEIGHTS)

for cam in CAMERAS:
    print(f"\n{'='*40}\nTracking {cam}...")
    
    # Get sorted images for this camera
    images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])
    print(f"  {len(images)} frames")
    
    # Fresh tracker per camera
    tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.3)
    
    results_rows = []
    
    for frame_idx, img_path in enumerate(images):
        # Detect
        dets = model(img_path, verbose=False)[0]
        
        # Format detections for DeepSORT: [[x1,y1,w,h], confidence, class]
        ds_dets = []
        for box in dets.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            w, h = x2 - x1, y2 - y1
            ds_dets.append(([x1, y1, w, h], conf, 'cow'))
        
        # Update tracker
        # Update tracker (needs image for appearance features)
        frame = cv2.imread(str(img_path))
        tracks = tracker.update_tracks(ds_dets, frame=frame)
        
        for track in tracks:
            if not track.is_confirmed():
                continue
            track_id = track.track_id
            ltrb = track.to_ltrb()  # left, top, right, bottom
            results_rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": ltrb[0], "y1": ltrb[1],
                "x2": ltrb[2], "y2": ltrb[3],
                "image": img_path.name
            })
    
    # Save per-camera tracklets
    out_csv = OUTPUT_DIR / f"{cam}_tracks.csv"
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["frame","track_id","x1","y1","x2","y2","image"])
        writer.writeheader()
        writer.writerows(results_rows)
    
    n_tracks = len(set(r["track_id"] for r in results_rows))
    print(f"  {len(results_rows)} detections, {n_tracks} unique track IDs")
    print(f"  Saved to {out_csv}")

print("\n✅ DeepSORT tracking complete")


Tracking cam_1...
  756 frames
  25990 detections, 283 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_1_tracks.csv

Tracking cam_2...
  755 frames
  23578 detections, 247 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_2_tracks.csv

Tracking cam_3...
  756 frames
  22668 detections, 255 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_3_tracks.csv

Tracking cam_4...
  756 frames
  23476 detections, 281 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_4_tracks.csv

✅ DeepSORT tracking complete


## Bot-Sort Tracking

In [8]:
# Cell 23: BoT-SORT tracking on test split
from ultralytics import YOLO
from pathlib import Path
import csv

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
OUTPUT_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/botsort")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]
model = YOLO(WEIGHTS)

for cam in CAMERAS:
    print(f"\n{'='*40}\nTracking {cam}...")
    
    images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])
    print(f"  {len(images)} frames")
    
    results_rows = []
    
    # YOLO built-in tracker — processes frame by frame with persist=True
    for frame_idx, img_path in enumerate(images):
        results = model.track(
            img_path,
            tracker="botsort.yaml",
            persist=True,
            verbose=False,
            device=0,
        )
        
        for box in results[0].boxes:
            if box.id is None:
                continue
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            track_id = int(box.id[0])
            results_rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": x1, "y1": y1,
                "x2": x2, "y2": y2,
                "image": img_path.name
            })
    
    # Save
    out_csv = OUTPUT_DIR / f"{cam}_tracks.csv"
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["frame","track_id","x1","y1","x2","y2","image"])
        writer.writeheader()
        writer.writerows(results_rows)
    
    n_tracks = len(set(r["track_id"] for r in results_rows))
    print(f"  {len(results_rows)} detections, {n_tracks} unique track IDs")
    print(f"  Saved to {out_csv}")
    
    # Reset tracker between cameras
    model.predictor = None

print("\n✅ BoT-SORT tracking complete")


Tracking cam_1...
  756 frames
  7765 detections, 235 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_1_tracks.csv

Tracking cam_2...
  755 frames
  2703 detections, 55 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_2_tracks.csv

Tracking cam_3...
  756 frames
  2447 detections, 50 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_3_tracks.csv

Tracking cam_4...
  756 frames
  2538 detections, 72 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_4_tracks.csv

✅ BoT-SORT tracking complete


In [9]:
# Cell 24: Check BoT-SORT default config
import ultralytics, yaml
from pathlib import Path

botsort_cfg = Path(ultralytics.__file__).parent / "cfg/trackers/botsort.yaml"
print(yaml.safe_load(botsort_cfg.read_text()))

{'tracker_type': 'botsort', 'track_high_thresh': 0.25, 'track_low_thresh': 0.1, 'new_track_thresh': 0.25, 'track_buffer': 30, 'match_thresh': 0.8, 'fuse_score': True, 'gmc_method': 'sparseOptFlow', 'proximity_thresh': 0.5, 'appearance_thresh': 0.8, 'with_reid': False, 'model': 'auto'}


In [12]:
# Cell 24b: Quick test — compare detections per frame
from ultralytics import YOLO
from pathlib import Path

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
model = YOLO(WEIGHTS)

images = sorted([f for f in YOLO_ROOT.glob("cam_2_*.jpg")])[:10]

print("=== Detection only (no tracking) ===")
for img in images[:5]:
    r = model(img, verbose=False)[0]
    print(f"  {img.name}: {len(r.boxes)} boxes")

print("\n=== BoT-SORT tracking ===")
for img in images[:5]:
    r = model.track(img, tracker="botsort.yaml", persist=True, verbose=False, device=0)[0]
    tracked = [b for b in r.boxes if b.id is not None]
    print(f"  {img.name}: {len(r.boxes)} boxes, {len(tracked)} tracked")

=== Detection only (no tracking) ===
  cam_2_1690272026_03-00-26.jpg: 11 boxes
  cam_2_1690272101_03-01-41.jpg: 11 boxes
  cam_2_1690272191_03-03-11.jpg: 11 boxes
  cam_2_1690272281_03-04-41.jpg: 11 boxes
  cam_2_1690272341_03-05-41.jpg: 11 boxes

=== BoT-SORT tracking ===
  cam_2_1690272026_03-00-26.jpg: 11 boxes, 11 tracked
  cam_2_1690272101_03-01-41.jpg: 10 boxes, 10 tracked
  cam_2_1690272191_03-03-11.jpg: 10 boxes, 10 tracked
  cam_2_1690272281_03-04-41.jpg: 10 boxes, 10 tracked
  cam_2_1690272341_03-05-41.jpg: 10 boxes, 10 tracked


In [13]:
# Cell 25: BoT-SORT tracking — FIXED
from ultralytics import YOLO
from pathlib import Path
import csv

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
OUTPUT_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/botsort")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]

for cam in CAMERAS:
    print(f"\n{'='*40}\nTracking {cam}...")
    
    # Fresh model per camera to fully reset tracker state
    model = YOLO(WEIGHTS)
    
    images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])
    print(f"  {len(images)} frames")
    
    results_rows = []
    
    for frame_idx, img_path in enumerate(images):
        results = model.track(
            img_path,
            tracker="botsort.yaml",
            persist=True,
            verbose=False,
            device=0,
        )
        
        for box in results[0].boxes:
            if box.id is None:
                continue
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            track_id = int(box.id[0])
            results_rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": x1, "y1": y1,
                "x2": x2, "y2": y2,
                "image": img_path.name
            })
    
    # Save
    out_csv = OUTPUT_DIR / f"{cam}_tracks.csv"
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["frame","track_id","x1","y1","x2","y2","image"])
        writer.writeheader()
        writer.writerows(results_rows)
    
    n_tracks = len(set(r["track_id"] for r in results_rows))
    print(f"  {len(results_rows)} detections, {n_tracks} unique track IDs")
    print(f"  Saved to {out_csv}")

print("\n✅ BoT-SORT tracking complete")


Tracking cam_1...
  756 frames
  7765 detections, 235 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_1_tracks.csv

Tracking cam_2...
  755 frames
  7986 detections, 228 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_2_tracks.csv

Tracking cam_3...
  756 frames
  7797 detections, 224 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_3_tracks.csv

Tracking cam_4...
  756 frames
  7482 detections, 230 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_4_tracks.csv

✅ BoT-SORT tracking complete


In [14]:
# Cell 26: Debug — detections per frame comparison
import pandas as pd

ds = pd.read_csv("/home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_2_tracks.csv")
bs = pd.read_csv("/home/oussema/pfa-mmcows/outputs/tracks/botsort/cam_2_tracks.csv")

print("=== DeepSORT cam_2 ===")
print(f"Total detections: {len(ds)}")
print(f"Dets per frame (mean): {ds.groupby('frame').size().mean():.1f}")
print(f"Dets per frame (first 5):\n{ds.groupby('frame').size().head()}")

print("\n=== BoT-SORT cam_2 ===")
print(f"Total detections: {len(bs)}")
print(f"Dets per frame (mean): {bs.groupby('frame').size().mean():.1f}")
print(f"Dets per frame (first 5):\n{bs.groupby('frame').size().head()}")

=== DeepSORT cam_2 ===
Total detections: 23578
Dets per frame (mean): 31.3
Dets per frame (first 5):
frame
2     9
3    10
4    10
5    11
6    12
dtype: int64

=== BoT-SORT cam_2 ===
Total detections: 7986
Dets per frame (mean): 11.2
Dets per frame (first 5):
frame
0    11
1    10
2    10
3    10
4    10
dtype: int64


In [15]:
# Cell 27: How many tracks per frame over time?
print("=== DeepSORT dets per frame (every 100 frames) ===")
ds_per_frame = ds.groupby('frame').size()
for f in [0, 5, 50, 100, 200, 400, 600, 750]:
    if f in ds_per_frame.index:
        print(f"  frame {f}: {ds_per_frame[f]} detections")

=== DeepSORT dets per frame (every 100 frames) ===
  frame 5: 11 detections
  frame 50: 29 detections
  frame 100: 29 detections
  frame 200: 36 detections
  frame 400: 33 detections
  frame 600: 34 detections
  frame 750: 27 detections


## Fixed deepsort tracking code (dont count desapearing cows)

In [16]:
# Cell 28: DeepSORT tracking — FIXED
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from pathlib import Path
import csv, cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
OUTPUT_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/deepsort")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]
model = YOLO(WEIGHTS)

for cam in CAMERAS:
    print(f"\n{'='*40}\nTracking {cam}...")
    
    images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])
    print(f"  {len(images)} frames")
    
    tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.3)
    results_rows = []
    
    for frame_idx, img_path in enumerate(images):
        dets = model(img_path, verbose=False)[0]
        
        ds_dets = []
        for box in dets.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            w, h = x2 - x1, y2 - y1
            ds_dets.append(([x1, y1, w, h], conf, 'cow'))
        
        frame = cv2.imread(str(img_path))
        tracks = tracker.update_tracks(ds_dets, frame=frame)
        
        for track in tracks:
            if not track.is_confirmed() or track.is_deleted():
                continue
            track_id = track.track_id
            ltrb = track.to_ltrb()
            results_rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": ltrb[0], "y1": ltrb[1],
                "x2": ltrb[2], "y2": ltrb[3],
                "image": img_path.name
            })
    
    out_csv = OUTPUT_DIR / f"{cam}_tracks.csv"
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["frame","track_id","x1","y1","x2","y2","image"])
        writer.writeheader()
        writer.writerows(results_rows)
    
    n_tracks = len(set(r["track_id"] for r in results_rows))
    print(f"  {len(results_rows)} detections, {n_tracks} unique track IDs")
    print(f"  Saved to {out_csv}")

print("\n✅ DeepSORT tracking complete (fixed)")


Tracking cam_1...
  756 frames
  25990 detections, 283 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_1_tracks.csv

Tracking cam_2...
  755 frames
  23578 detections, 247 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_2_tracks.csv

Tracking cam_3...
  756 frames
  22668 detections, 255 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_3_tracks.csv

Tracking cam_4...
  756 frames
  23476 detections, 281 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_4_tracks.csv

✅ DeepSORT tracking complete (fixed)


In [17]:
# Cell 29: Debug track states
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from pathlib import Path
import cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
model = YOLO(WEIGHTS)

images = sorted([f for f in YOLO_ROOT.glob("cam_2_*.jpg")])
tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.3)

# Run 50 frames, check states at frame 50
for i, img_path in enumerate(images[:50]):
    dets = model(img_path, verbose=False)[0]
    ds_dets = []
    for box in dets.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        w, h = x2 - x1, y2 - y1
        ds_dets.append(([x1, y1, w, h], conf, 'cow'))
    
    frame = cv2.imread(str(img_path))
    tracks = tracker.update_tracks(ds_dets, frame=frame)
    
    if i == 49:
        print(f"Frame {i}: {len(tracks)} total tracks")
        confirmed_alive = 0
        confirmed_deleted = 0
        tentative = 0
        for t in tracks:
            if t.is_confirmed() and not t.is_deleted():
                confirmed_alive += 1
            elif t.is_confirmed() and t.is_deleted():
                confirmed_deleted += 1
            else:
                tentative += 1
        print(f"  Confirmed & alive: {confirmed_alive}")
        print(f"  Confirmed & deleted: {confirmed_deleted}")
        print(f"  Tentative: {tentative}")

Frame 49: 31 total tracks
  Confirmed & alive: 29
  Confirmed & deleted: 0
  Tentative: 2


In [18]:
# Cell 30: Check what attributes tracks have
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from pathlib import Path
import cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
model = YOLO(WEIGHTS)

images = sorted([f for f in YOLO_ROOT.glob("cam_2_*.jpg")])
tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.3)

for i, img_path in enumerate(images[:50]):
    dets = model(img_path, verbose=False)[0]
    ds_dets = []
    for box in dets.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        w, h = x2 - x1, y2 - y1
        ds_dets.append(([x1, y1, w, h], conf, 'cow'))
    frame = cv2.imread(str(img_path))
    tracks = tracker.update_tracks(ds_dets, frame=frame)

    if i == 49:
        print(f"Frame {i}: {len(tracks)} tracks")
        for t in tracks[:5]:
            print(f"  ID={t.track_id}, confirmed={t.is_confirmed()}, "
                  f"time_since_update={t.time_since_update}, "
                  f"age={t.age}")

Frame 49: 31 tracks
  ID=1, confirmed=True, time_since_update=0, age=50
  ID=6, confirmed=True, time_since_update=0, age=50
  ID=8, confirmed=True, time_since_update=0, age=50
  ID=9, confirmed=True, time_since_update=28, age=50
  ID=11, confirmed=True, time_since_update=0, age=50


In [1]:
# Cell 31: DeepSORT tracking — PROPERLY FIXED
#
# What changed from before:
# OLD filter: if not track.is_confirmed() or track.is_deleted()
# NEW filter: if not track.is_confirmed() or track.time_since_update > 0
#
# time_since_update = how many frames since this track last matched a real detection
# 0 means "matched this frame" = real cow visible now
# >0 means "not seen for N frames" = ghost prediction, skip it

from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from pathlib import Path
import csv, cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
OUTPUT_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/deepsort")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]
model = YOLO(WEIGHTS)

for cam in CAMERAS:
    print(f"\n{'='*40}\nTracking {cam}...")
    
    images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])
    print(f"  {len(images)} frames")
    
    tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.3)
    results_rows = []
    
    for frame_idx, img_path in enumerate(images):
        dets = model(img_path, verbose=False)[0]
        
        ds_dets = []
        for box in dets.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            w, h = x2 - x1, y2 - y1
            ds_dets.append(([x1, y1, w, h], conf, 'cow'))
        
        frame = cv2.imread(str(img_path))
        tracks = tracker.update_tracks(ds_dets, frame=frame)
        
        for track in tracks:
            # ONLY keep tracks that are confirmed AND matched this frame
            if not track.is_confirmed() or track.time_since_update > 0:
                continue
            track_id = track.track_id
            ltrb = track.to_ltrb()
            results_rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": ltrb[0], "y1": ltrb[1],
                "x2": ltrb[2], "y2": ltrb[3],
                "image": img_path.name
            })
    
    out_csv = OUTPUT_DIR / f"{cam}_tracks.csv"
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["frame","track_id","x1","y1","x2","y2","image"])
        writer.writeheader()
        writer.writerows(results_rows)
    
    n_tracks = len(set(r["track_id"] for r in results_rows))
    print(f"  {len(results_rows)} detections, {n_tracks} unique track IDs")
    print(f"  Saved to {out_csv}")

print("\n✅ DeepSORT tracking complete (fixed)")


Tracking cam_1...
  756 frames


/opt/miniconda3/lib/python3.13/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


  7274 detections, 283 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_1_tracks.csv

Tracking cam_2...
  755 frames
  7640 detections, 247 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_2_tracks.csv

Tracking cam_3...
  756 frames
  7443 detections, 255 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_3_tracks.csv

Tracking cam_4...
  756 frames
  6974 detections, 281 unique track IDs
  Saved to /home/oussema/pfa-mmcows/outputs/tracks/deepsort/cam_4_tracks.csv

✅ DeepSORT tracking complete (fixed)


## Evaluation

In [2]:
# Cell 33: Build ground truth from original labels + evaluate both trackers
import motmetrics as mm
import pandas as pd
import numpy as np
from pathlib import Path

LABELS_ORIG = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data/labels/combined/0725")
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
DS_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/deepsort")
BS_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/botsort")
CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]


In [3]:

def load_gt_for_camera(cam):
    """
    Load ground truth from original labels.
    Original format: cow_id xc yc w h (normalized 0-1)
    We need: frame_index, cow_id, x1, y1, x2, y2 (pixels)
    """
    images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])
    rows = []
    for frame_idx, img_path in enumerate(images):
        # Get original label (without cam_ prefix)
        orig_stem = img_path.stem.replace(f"{cam}_", "")
        lbl_path = LABELS_ORIG / cam / (orig_stem + ".txt")
        
        if not lbl_path.exists():
            continue
        
        # Read image size for denormalization
        import cv2
        img = cv2.imread(str(img_path))
        h_img, w_img = img.shape[:2]
        
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cow_id = int(parts[0])
                xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                
                # Convert normalized xywh → pixel x1y1x2y2
                x1 = (xc - w/2) * w_img
                y1 = (yc - h/2) * h_img
                x2 = (xc + w/2) * w_img
                y2 = (yc + h/2) * h_img
                
                rows.append({
                    "frame": frame_idx,
                    "cow_id": cow_id,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2
                })
    return pd.DataFrame(rows)



In [5]:
# Cell 33a: Fix numpy compatibility
import numpy as np
# Patch the removed function
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

print("✅ Patched np.asfarray")

✅ Patched np.asfarray


In [7]:

def evaluate_tracker(gt_df, tracker_df):
    """
    Compare tracker output against ground truth using motmetrics.
    
    For each frame:
    - gt has cow_id + bbox
    - tracker has track_id + bbox
    - We compute IoU (overlap) between all pairs
    - motmetrics matches them and counts: hits, misses, false alarms, ID switches
    """
    acc = mm.MOTAccumulator(auto_id=True)
    
    all_frames = sorted(set(gt_df["frame"].unique()) | set(tracker_df["frame"].unique()))
    
    for frame in all_frames:
        gt_frame = gt_df[gt_df["frame"] == frame]
        tr_frame = tracker_df[tracker_df["frame"] == frame]
        
        gt_ids = gt_frame["cow_id"].tolist()
        tr_ids = tr_frame["track_id"].tolist()
        
        # Compute IoU distance matrix between GT and tracker boxes
        if len(gt_ids) > 0 and len(tr_ids) > 0:
            gt_boxes = gt_frame[["x1","y1","x2","y2"]].values
            tr_boxes = tr_frame[["x1","y1","x2","y2"]].values
            
            # mm.distances.iou_matrix gives 1-IoU (distance), NaN if no overlap
            dists = mm.distances.iou_matrix(gt_boxes, tr_boxes, max_iou=0.5)
        else:
            dists = np.empty((len(gt_ids), len(tr_ids)))
        
        acc.update(gt_ids, tr_ids, dists)
    
    return acc


# === RUN EVALUATION ===
all_results = []

for cam in CAMERAS:
    print(f"\nEvaluating {cam}...")
    
    gt_df = load_gt_for_camera(cam)
    ds_df = pd.read_csv(DS_DIR / f"{cam}_tracks.csv")
    bs_df = pd.read_csv(BS_DIR / f"{cam}_tracks.csv")
    
    ds_acc = evaluate_tracker(gt_df, ds_df)
    bs_acc = evaluate_tracker(gt_df, bs_df)
    
    # Compute metrics
    mh = mm.metrics.create()
    ds_metrics = mh.compute(ds_acc, metrics=["mota","motp","idf1","num_switches","precision","recall"], name=f"DeepSORT_{cam}")
    bs_metrics = mh.compute(bs_acc, metrics=["mota","motp","idf1","num_switches","precision","recall"], name=f"BoT-SORT_{cam}")
    
    print(f"\n  DeepSORT: MOTA={ds_metrics['mota'].values[0]:.4f}, MOTP={ds_metrics['motp'].values[0]:.4f}, IDF1={ds_metrics['idf1'].values[0]:.4f}, IDsw={int(ds_metrics['num_switches'].values[0])}")
    print(f"  BoT-SORT: MOTA={bs_metrics['mota'].values[0]:.4f}, MOTP={bs_metrics['motp'].values[0]:.4f}, IDF1={bs_metrics['idf1'].values[0]:.4f}, IDsw={int(bs_metrics['num_switches'].values[0])}")
    
    all_results.append(ds_metrics)
    all_results.append(bs_metrics)

# === SUMMARY TABLE ===
print("\n" + "="*60)
print("FULL COMPARISON")
print("="*60)
summary = pd.concat(all_results)
print(summary.to_string())


Evaluating cam_1...

  DeepSORT: MOTA=0.7097, MOTP=0.0713, IDF1=0.1278, IDsw=1226
  BoT-SORT: MOTA=0.8023, MOTP=0.0845, IDF1=0.1620, IDsw=980

Evaluating cam_2...

  DeepSORT: MOTA=0.7402, MOTP=0.0960, IDF1=0.1777, IDsw=1112
  BoT-SORT: MOTA=0.8074, MOTP=0.0946, IDF1=0.1988, IDsw=930

Evaluating cam_3...

  DeepSORT: MOTA=0.6993, MOTP=0.0909, IDF1=0.1783, IDsw=1134
  BoT-SORT: MOTA=0.7759, MOTP=0.0876, IDF1=0.2002, IDsw=968

Evaluating cam_4...

  DeepSORT: MOTA=0.6920, MOTP=0.0922, IDF1=0.1658, IDsw=1167
  BoT-SORT: MOTA=0.7812, MOTP=0.0904, IDF1=0.1781, IDsw=979

FULL COMPARISON
                    mota      motp      idf1  num_switches  precision    recall
DeepSORT_cam_1  0.709725  0.071285  0.127765          1226   0.977042  0.882747
BoT-SORT_cam_1  0.802261  0.084457  0.161988           980   0.979008  0.944231
DeepSORT_cam_2  0.740247  0.095952  0.177677          1112   0.972644  0.900291
BoT-SORT_cam_2  0.807366  0.094642  0.198768           930   0.975457  0.943785
DeepSORT_ca

## Fine-tuning Bot-Sort

In [8]:
# Cell 34: BoT-SORT hyperparameter tuning
#
# We test different combinations of parameters on cam_2 (representative camera)
# and pick the combo with best MOTA + IDF1
#
# Instead of editing botsort.yaml, we create temporary YAML files per combo

import yaml, os, shutil
import numpy as np
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

import motmetrics as mm
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import csv, cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
LABELS_ORIG = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data/labels/combined/0725")
TUNE_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/botsort_tune")
TUNE_DIR.mkdir(parents=True, exist_ok=True)

# === Grid of parameters to test ===
param_grid = [
    {"track_buffer": 30, "match_thresh": 0.8, "track_high_thresh": 0.25},  # default
    {"track_buffer": 50, "match_thresh": 0.8, "track_high_thresh": 0.25},  # longer memory
    {"track_buffer": 30, "match_thresh": 0.6, "track_high_thresh": 0.25},  # easier matching
    {"track_buffer": 50, "match_thresh": 0.6, "track_high_thresh": 0.25},  # both
    {"track_buffer": 50, "match_thresh": 0.6, "track_high_thresh": 0.5},   # stricter init
    {"track_buffer": 70, "match_thresh": 0.7, "track_high_thresh": 0.3},   # long memory + balanced
]

# === Run grid search on cam_2 ===
cam = "cam_2"
gt_df = load_gt_for_camera(cam)
images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])

print(f"Tuning on {cam} ({len(images)} frames)\n")
print(f"{'Config':<55} {'MOTA':>7} {'MOTP':>7} {'IDF1':>7} {'IDsw':>6}")
print("-" * 85)

best_score = -1
best_params = None

for i, params in enumerate(param_grid):
    # Create custom YAML
    cfg = {
        "tracker_type": "botsort",
        "track_high_thresh": params["track_high_thresh"],
        "track_low_thresh": 0.1,
        "new_track_thresh": params["track_high_thresh"],
        "track_buffer": params["track_buffer"],
        "match_thresh": params["match_thresh"],
        "fuse_score": True,
        "gmc_method": "sparseOptFlow",
        "proximity_thresh": 0.5,
        "appearance_thresh": 0.8,
        "with_reid": False,
        "model": "auto",
    }
    cfg_path = TUNE_DIR / f"botsort_v{i}.yaml"
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f)
    
    # Track
    model = YOLO(WEIGHTS)
    results_rows = []
    for frame_idx, img_path in enumerate(images):
        results = model.track(img_path, tracker=str(cfg_path), persist=True, verbose=False, device=0)
        for box in results[0].boxes:
            if box.id is None: continue
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            track_id = int(box.id[0])
            results_rows.append({"frame": frame_idx, "track_id": track_id, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    
    tr_df = pd.DataFrame(results_rows)
    n_ids = len(tr_df["track_id"].unique()) if len(tr_df) > 0 else 0
    
    # Evaluate
    acc = evaluate_tracker(gt_df, tr_df)
    mh = mm.metrics.create()
    metrics = mh.compute(acc, metrics=["mota","motp","idf1","num_switches"])
    
    mota = metrics["mota"].values[0]
    motp = metrics["motp"].values[0]
    idf1 = metrics["idf1"].values[0]
    idsw = int(metrics["num_switches"].values[0])
    
    label = f"buf={params['track_buffer']}, match={params['match_thresh']}, high={params['track_high_thresh']}"
    print(f"{label:<55} {mota:>7.4f} {motp:>7.4f} {idf1:>7.4f} {idsw:>6}")
    
    # Score = MOTA + IDF1 (we want both high)
    score = mota + idf1
    if score > best_score:
        best_score = score
        best_params = params

print(f"\n✅ Best config: {best_params}")
print(f"   Combined score (MOTA + IDF1): {best_score:.4f}")

Tuning on cam_2 (755 frames)

Config                                                     MOTA    MOTP    IDF1   IDsw
-------------------------------------------------------------------------------------
buf=30, match=0.8, high=0.25                             0.8074  0.0946  0.1988    930
buf=50, match=0.8, high=0.25                             0.8100  0.0921  0.2032    969
buf=30, match=0.6, high=0.25                             0.7478  0.0641  0.1701   1065
buf=50, match=0.6, high=0.25                             0.7502  0.0687  0.1729   1108
buf=50, match=0.6, high=0.5                              0.7508  0.0671  0.1815   1032
buf=70, match=0.7, high=0.3                              0.7894  0.0802  0.1916   1036

✅ Best config: {'track_buffer': 50, 'match_thresh': 0.8, 'track_high_thresh': 0.25}
   Combined score (MOTA + IDF1): 1.0132


In [9]:
# Cell 35: Test BoT-SORT with Re-ID enabled
#
# with_reid=True tells BoT-SORT to extract appearance features
# from each cow crop and use them for matching.
# Think of it like: instead of just tracking by position,
# the tracker also looks at "what does this cow look like?"
# to help match it across frames.

import yaml
import numpy as np
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

import motmetrics as mm
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import cv2

WEIGHTS = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt"
YOLO_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test")
LABELS_ORIG = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data/labels/combined/0725")
TUNE_DIR = Path("/home/oussema/pfa-mmcows/outputs/tracks/botsort_tune")

# Create config with Re-ID enabled
cfg = {
    "tracker_type": "botsort",
    "track_high_thresh": 0.25,
    "track_low_thresh": 0.1,
    "new_track_thresh": 0.25,
    "track_buffer": 30,
    "match_thresh": 0.8,
    "fuse_score": True,
    "gmc_method": "sparseOptFlow",
    "proximity_thresh": 0.5,
    "appearance_thresh": 0.8,
    "with_reid": True,       # <-- THIS IS THE CHANGE
    "model": "auto",         # auto-downloads a Re-ID model
}
cfg_path = TUNE_DIR / "botsort_reid.yaml"
with open(cfg_path, "w") as f:
    yaml.dump(cfg, f)
    
# --- Run on cam_2 ---
cam = "cam_2"
gt_df = load_gt_for_camera(cam)
images = sorted([f for f in YOLO_ROOT.glob(f"{cam}_*.jpg")])

model = YOLO(WEIGHTS)
results_rows = []

print(f"Running BoT-SORT + Re-ID on {cam} ({len(images)} frames)...")
for frame_idx, img_path in enumerate(images):
    results = model.track(img_path, tracker=str(cfg_path), persist=True, verbose=False, device=0)
    for box in results[0].boxes:
        if box.id is None: continue
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        track_id = int(box.id[0])
        results_rows.append({"frame": frame_idx, "track_id": track_id,
                             "x1": x1, "y1": y1, "x2": x2, "y2": y2})

tr_df = pd.DataFrame(results_rows)
n_ids = len(tr_df["track_id"].unique())

acc = evaluate_tracker(gt_df, tr_df)
mh = mm.metrics.create()
m = mh.compute(acc, metrics=["mota","motp","idf1","num_switches","precision","recall"])

print(f"\n{'='*50}")
print(f"BoT-SORT + Re-ID results on {cam}:")
print(f"  MOTA:  {m['mota'].values[0]:.4f}  (without Re-ID: 0.8074)")
print(f"  MOTP:  {m['motp'].values[0]:.4f}  (without Re-ID: 0.0946)")
print(f"  IDF1:  {m['idf1'].values[0]:.4f}  (without Re-ID: 0.1988)")
print(f"  IDsw:  {int(m['num_switches'].values[0])}      (without Re-ID: 930)")
print(f"  IDs:   {n_ids}       (without Re-ID: 228)")

Running BoT-SORT + Re-ID on cam_2 (755 frames)...

BoT-SORT + Re-ID results on cam_2:
  MOTA:  0.7989  (without Re-ID: 0.8074)
  MOTP:  0.1027  (without Re-ID: 0.0946)
  IDF1:  0.1925  (without Re-ID: 0.1988)
  IDsw:  959      (without Re-ID: 930)
  IDs:   251       (without Re-ID: 228)
